# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya — Exploration with `mlcroissant`
This notebook demonstrates step-by-step how to explore and process a Croissant dataset using the `mlcroissant` Python library, referencing all dataset entities by their `@id`.

### Dataset Source
FAIR⁲ dataset is defined by a Croissant schema accessible here:

https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
# Access single metadata object (avoid subscripting/iteration)
meta = dataset.metadata

print(f"{meta.name}: {meta.description}")
print(f"License: {meta.license}")
print(f"Keywords: {meta.keywords}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# The mlcroissant Dataset provides 'record_sets' — let's list all record sets and fields by @id.

# List all record sets
record_sets = list(dataset.record_sets)
print("Available Record Sets and their fields (by @id):")

for record_set in record_sets:
    print(f"- RecordSet @id: {record_set['@id']}")
    if 'field' in record_set:
        fields = record_set['field']
        if isinstance(fields, dict):
            fields = [fields]
        print("  Fields:")
        for field in fields:
            # Each field is a dict with '@id'.
            print(f"    - {field['@id']}")
    else:
        print("  (No fields found)")
    print()

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.
Use the record set and field `@id`s found above.

In [ ]:
# Extract data from each record set into Pandas DataFrames.

# Get record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)

# Display columns of each DataFrame
for rs_id, df in dataframes.items():
    print(f"RecordSet {rs_id}: columns=")
    print(df.columns.tolist())
    print(df.head(3))
    print()

# For demonstration: pick the first populated record set (if any)
if dataframes:
    example_rs_id = list(dataframes.keys())[0]
    print(f"Using RecordSet {example_rs_id} for analysis.")
    example_df = dataframes[example_rs_id]
else:
    example_rs_id = None
    print('No record sets contained tabular data.')

## 4. Exploratory Data Analysis (EDA)
Apply processing steps—such as filtering records, normalizing fields, grouping—referencing fields by their `@id`.

In [ ]:
# This block demonstrates EDA on a numeric field from a populated record set.
import numpy as np

if example_rs_id is not None:
    df = dataframes[example_rs_id]

    # Find the first numeric column (for demo), else specify based on dataset docs
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    if numeric_cols:
        numeric_field_id = numeric_cols[0]  # Use first numeric column's @id
        print(f"Selected numeric field (@id): {numeric_field_id}")

        # Filter: keep records where this field > threshold (example: use 10 or median)
        threshold = df[numeric_field_id].median()
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head(3))

        # Normalize the numeric field
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, norm_col]].head(3))

        # Group by a candidate categorical column if exists
        group_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
        # Skip very high-cardinality like index columns
        group_col = None
        for col in group_cols:
            if df[col].nunique() < len(df) / 2 and col != numeric_field_id:
                group_col = col
                break
        if group_col:
            grouped = filtered_df.groupby(group_col)[numeric_field_id].mean()
            print(f"Mean {numeric_field_id}, grouped by {group_col}:")
            print(grouped.head())
    else:
        print("No numeric fields found in this record set.")
else:
    print("No populated record set available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Basic Histograms/Scatter if data is available:
import matplotlib.pyplot as plt

if example_rs_id is not None and numeric_cols:
    # Histogram of the selected numeric field
    plt.figure(figsize=(7, 4))
    df[numeric_field_id].hist(bins=20, color='skyblue')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.title(f'Histogram of {numeric_field_id}')
    plt.show()

    # If grouping column exists (from EDA above), show boxplot
    if 'group_col' in locals() and group_col:
        plt.figure(figsize=(8,4))
        df.boxplot(column=numeric_field_id, by=group_col)
        plt.title(f'{numeric_field_id} by {group_col}')
        plt.suptitle("")
        plt.xlabel(group_col)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
This notebook demonstrated loading a Croissant dataset, examining metadata and structure via `@id`, extracting and inspecting tabular records, and applying basic processing and visualization with `mlcroissant`.

- Records, fields, and columns were accessed and referenced by their Croissant `@id`s.
- You can customize further steps and analyses by exploring more fields as needed.

Explore more about [mlcroissant](https://mlcommons.github.io/croissant/) and [FAIR² datasets](https://sen.science/)!